## Software Design Patterns in Python
### Day 1: Python OOP Refresher & The Data Model

---

#### Module 1: Advanced Object Mechanics

- The Object Lifecycle: Deep dive into `__new__` (allocation) vs `__init__` (initialization).
- Metaclasses: Practical use cases (registry pattern, enforcing interface constraints).
- Memory Optimization: Using `__slots__` and weakref for memory-efficient object design.

---

#### Module 2: The Python Data Model & Protocols

- Descriptor Protocol: Implementing `__get__`, `__set__`, and `__getattr__` / `__setattr__` / `__delattr__` (The foundation of Python properties).
- Property accessors: Using the builtin `@property` accessors.
- Context Management: Creating custom context managers with `__enter__` and `__exit__`.
- Callable Objects: Using `__call__` to make instances behave like functions.
- Review: How these hooks replace "boilerplate" code found in Java/C++ patterns.

---



### Preliminary Exercise:

- Download the exercise file from https://tinyurl.com/pyshell-deco-ex 
- Read through the `docstring` and implement the `CommandDispatch` class in `command_dispatch` module
  such that the code in the exercise file works.

In [ ]:
# %load https://tinyurl.com/pyshell-deco-ex
"""
Exercise:
     Implement the command_dispatch module with class CommandDispatch
     such that the following example program that implements a
     rudimentary command-line shell works.
"""

from command_dispatch import CommandDispatch


shell = CommandDispatch()

@shell.for_command("list")
def list_directory(*args):
    from os import listdir
    if len(args) < 2:
        args += ".",
    for path in args[1:]:
        print("{}:".format(path))
        print("\n".join(listdir(path)))

@shell.for_command("whoami")
def show_user(*args):
    from getpass import getuser
    print(getuser())

@shell.for_command("date")
def print_date(*args):
    from time import ctime
    print(ctime())

@shell.for_command("pwd")
def show_curr_dir(*args):
    import os
    print(os.getcwd())

@shell.for_command("exit")
def exit_shell(*args):
    exit(0)

@shell.for_command("hostname")
def show_hostname(*args):
    from os import uname
    print(uname().nodename)

@shell.invalid
def invalid_command(*args):
    print("Invalid command - ", args[0])


@shell.input
def get_input():
    import rlcompleter
    return input("PyShell> ").split()

if __name__ == '__main__':
    shell.run()



In [1]:
%save pyshell.py 1

Operation cancelled.


In [ ]:
class User:
    def __init__(self):
        print("Init method called")
        self.name = "Chandra"

u = User()  # Constructor expression
u.name

Init method called


'Chandra'

In [3]:
u = User()

In [8]:
class User: pass

u1 = User()
u2 = User()
print(u1)
print(u2)

In [11]:
class User:
    def __new__(cls):
        print("New method called")
        return 100

u = User()
print(u)

New method called
100


In [10]:
#class User: pass
User = type("User", (), {})
u = User()
print(u)

In [16]:
class User:
    def __new__(cls):
        print("New method called: cls = ", cls)
        return super().__new__(cls)
        #return cls()

    def __init__(self):
        print("Init method called")
        self.name = "Chandra"

u = User()

New method called: cls =  <class '__main__.User'>
Init method called


In [22]:
class User:
    count = 0
    def __init__(self, name):
        self.name = name

    def greet(self):
        print(self.name, "says Hello")


u1 = User("Chandra")
u2 = User("Alice")
print(u1.name)
print(u2.name)
print(u1.count)
print(u2.count)
User.count = 100
print(u1.count)
print(u2.count)

User.greet(u1)
u1.greet()


Chandra
Alice
0
0
100
100
Chandra says Hello
Chandra says Hello


In [23]:
class User:
    print("Hello from User class")
    a = 100
    b = a + 20
    print(a, b)



Hello from User class
100 120


In [24]:
User.a, User.b

(100, 120)

In [25]:
class User: pass

print(User)

<class '__main__.User'>


In [30]:
print(User.__name__)
print(User.__dict__, vars(User), sep='\n')
print(User.__base__)

User
{'__module__': '__main__', '__dict__': <attribute '__dict__' of 'User' objects>, '__weakref__': <attribute '__weakref__' of 'User' objects>, '__doc__': None}
{'__module__': '__main__', '__dict__': <attribute '__dict__' of 'User' objects>, '__weakref__': <attribute '__weakref__' of 'User' objects>, '__doc__': None}
<class 'object'>


In [32]:
class Employee(User):
    pass

print(Employee.__name__)
print(Employee.__base__)
print(Employee.__bases__)

Employee
<class '__main__.User'>
(<class '__main__.User'>,)


In [34]:
a = 10
b = type(a)
print(b, type(b))

<class 'int'> <class 'type'>


NOTE: ```type``` vs ```object```

In Python, every class is an instance of a ```type```

However, every class inherit ```object```

In essence, ```type``` is a metaclass. Metaclasses are classes that create other classes


In [35]:
class User: pass

isinstance(User, type)

True

In [36]:
class Administrator:
    def __init__(self, name):
        self.name = name

class Employee:
    def __init__(self, name):
        self.name = name

class Guest:
    def __init__(self, name):
        self.name = name


class User:
    def __new__(cls, name):
        if name in ("root", "admin"):
            return Administrator(name)
        elif name in ("alice", "bob", "charlie"):
            return Employee(name)
        else:
            return Guest(name)
        
u1 = User("root")
u2 = User("alice")
u3 = User("eve")
print(type(u1), u1.name)
print(type(u2), u2.name)
print(type(u3), u3.name)

<class '__main__.Administrator'> root
<class '__main__.Employee'> alice
<class '__main__.Guest'> eve


In [39]:
class User(type): # User now is a custom metaclass
    pass

a = User("Administrator", (), {})
b = User("Employee", (), {})
print(a, type(a))
print(b, type(b))

a1 = a()
b1 = b()
print(a1, type(a1))
print(b1, type(b1))

<class '__main__.Administrator'> <class '__main__.User'>
<class '__main__.Employee'> <class '__main__.User'>
<__main__.Administrator object at 0x110b655e0> <class '__main__.Administrator'>
<__main__.Employee object at 0x110b66570> <class '__main__.Employee'>


In [40]:
class RegistryMeta(type):
    def __new__(cls, name, bases, namespace):
        print(f"cls={cls}, name={name}, bases={bases}, namespace={namespace}")

r = RegistryMeta("User", (), {"name": "Chandra"})
r

cls=<class '__main__.RegistryMeta'>, name=User, bases=(), namespace={'name': 'Chandra'}


In [43]:
class RegistryMeta(type):
    registry = {}

    def __new__(cls, name, bases, namespace):
        new_cls = super().__new__(cls, name, bases, namespace)
        if name != "Plugin":
            cls.registry[name] = new_cls
        return new_cls

    @classmethod
    def reload(cls):
        print("Reloading plugins...")
        for name, plugin_cls in cls.registry.items():
            print(f"Reloading plugin: {name} - {plugin_cls}")

class Plugin(metaclass=RegistryMeta):
    pass

class EmailPlugin(Plugin):
    pass

class SMSPlugin(Plugin):
    pass

class ChatPlugin(Plugin):
    pass

print(RegistryMeta.registry)

Plugin.reload()


{'EmailPlugin': <class '__main__.EmailPlugin'>, 'SMSPlugin': <class '__main__.SMSPlugin'>, 'ChatPlugin': <class '__main__.ChatPlugin'>}
Reloading plugins...
Reloading plugin: EmailPlugin - <class '__main__.EmailPlugin'>
Reloading plugin: SMSPlugin - <class '__main__.SMSPlugin'>
Reloading plugin: ChatPlugin - <class '__main__.ChatPlugin'>


In [50]:
class User: pass

u = User()

u.name = "John"
u.role = "Admin"
u.place = "New York"

print(u)
print(u.__dict__)

del u.role
print(u.__dict__)

#u.name
u.__dict__.get("name", u.__class__.__dict__.get("name", u.__class__.__bases__[0].__dict__.get("name", "Unknown")))

{'name': 'John', 'role': 'Admin', 'place': 'New York'}
{'name': 'John', 'place': 'New York'}


'John'

In [52]:
class Point:
    def __init__(self, x, y):
        self.x = x
        self.y = y

p = Point(10, 20)
print(p.__dict__)

{'x': 10, 'y': 20}


In [64]:
class Point:
    __slots__ = ['x', 'y']
    def __init__(self, x, y):
        self.x = x
        self.y = y

    def __str__(self):
        return f"Point(x={self.x}, y={self.y})"

p = Point(10, 20)
print(p)
print(p.x, p.y)
p.x = 30
print(p)
del p.y
# print(p.__dict__)  # This will raise an AttributeError because __slots__ is used
print(p.__slots__)
p.y = 100
print(p)
p.z = 200
print(p)

Point(x=10, y=20)
10 20
Point(x=30, y=20)
['x', 'y']
Point(x=30, y=100)


AttributeError: 'Point' object has no attribute 'z'

In [68]:
a = [10, 20, 30, 40, 50, 60, 70, 80, 90, 100] * 100
import sys
print(sys.getsizeof(a))

from array import array
b = array('b', a)
#print(b)
print(sys.getsizeof(b))

8056
1080


In [77]:
a = list(range(1000000))
b = set(a)

In [ ]:
%timeit 999 in a
%timeit 999 in b

2.2 μs ± 35.6 ns per loop (mean ± std. dev. of 7 runs, 100,000 loops each)
10.7 ns ± 0.0183 ns per loop (mean ± std. dev. of 7 runs, 100,000,000 loops each)


In [78]:
%timeit 999999 in a
%timeit 999999 in b

2.32 ms ± 20.4 μs per loop (mean ± std. dev. of 7 runs, 100 loops each)
10.8 ns ± 0.093 ns per loop (mean ± std. dev. of 7 runs, 100,000,000 loops each)


In [81]:
sys.getsizeof(a) / (1024 * 1024)

7.629447937011719

In [82]:
sys.getsizeof(b) / (1024 * 1024)

32.000205993652344

In [84]:
del User

In [ ]:
class User:
    def __init__(self, name):
        self.name = name

    def __repr__(self):
        return f"User(name='{self.name}')"
    
    def __str__(self):
        return f"<User name: {self.name}>"
    
u1 = User("Chandra")
print(u1)
print(str(u1), repr(u1))


User(name='Chandra')
User(name='Chandra') User(name='Chandra')


In [93]:
import httpx
response = httpx.get("https://www.python.org")
print(response)

<Response [200 OK]>


In [94]:
from urllib.request import urlopen
response = urlopen("https://www.python.org")
print(response)

In [ ]:
class Car:
    def __init__(self, make):
        self.make = make

    def __str__(self):
        return f"{self.make}"
    
    def __add__(self, other):
        return Car(f"{self.make} {other.make}")
    
    
c1 = Car("Maruti")
c2 = Car("Suzuki")
c3 = c1 + c2 # c1.__add__(c2) -> Car.__add__(c1, c2)
print(c3)

Maruti Suzuki


#### Exercise for today (Day 1):
 Create a command execution pipeline.
```
 import sys
 Run("ls") | Run("wc -l") > sys.stdout 
```

The above code should mimic the unix pipeline: ls | wc -l
For running commands and capturing output: use `subprocess.Popen`
